**Лабораторная работа №2. Информативные признаки речевых сигналов: извлечение признаков**

**Цель работы:** изучение процедуры построения информативных акустических признаков для речевых сигналов.

**Краткое описание**: в рамках настоящей лабораторной работы требуется познакомиться с процедурами предобработки речевых сигналов и извлечения информативных признаков. В работе предлагается научиться извлекать кратковременные энергии мел-частотных полос (MFBs, mel filter banks) и мел-частотные кепстральные коэффициенты (MFCCs, mel frequency cepstral coeffitients).

**Варианты**: id10276, id10288.

**Данные**: используется тестовая часть базы [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы:**

1. Подготовка данных для анализа.
2. Выполнение процедуры высокочастотной фильтрации (преэмфазис) речевого сигнала.
3. Вычисление акустических признаков разных видов.
4. Выполнение локальных центрирования и масштабирования акустических признаков.
5. Построение распределений первых трёх компонент полученных акустических признаков для мужских и женских голосов.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

_LAB_DIR = os.getcwd()
if not os.path.isfile(os.path.join(_LAB_DIR, "exercises_blank.py")):
    raise RuntimeError
_DATA_DIR = os.path.join(_LAB_DIR, "data")
_META_PATH = os.path.join(_LAB_DIR, "metadata", "meta.txt")
_DATASETS_LIST = os.path.join(_LAB_DIR, "data", "lists", "datasets.txt")

sys.path.insert(0, _LAB_DIR)

from common import download_dataset, extract_dataset

from math import sqrt, pi
from scipy.fftpack import dct

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import hist, plot, show, grid, title, xlabel, ylabel, legend, axis, imshow

from mpl_toolkits.axes_grid1 import make_axes_locatable

from skimage.morphology import opening, closing
from torchaudio.transforms import Resample
from multiprocessing import Pool
import torchaudio

from exercises_blank import split_meta_line, preemphasis, framing, power_spectrum
from exercises_blank import compute_fbank_filters, compute_fbanks_features, compute_mfcc, mvn_floating


**1. Подготовка данных для анализа**

В ходе выполнения лабораторной работы необходимы данные для осуществления процедуры вычисления акустических признаков. Возьмём в качестве этих данных несколько звукозаписей голосов людей мужского и женского пола, сохраненных в формат *wav*, выбранных из корпуса [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Данный корпус содержит 4,874 звукозаписи (частота дискретизации равна 16кГц) 40 дикторов мужского и женского пола, разговаривающих на английском языке.

В рамках настоящего пункта требуется выполнить загрузку и распаковку звуковых wav-файлов из корпуса VoxCeleb1 test set.

![Рисунок 1](https://analyticsindiamag.com/wp-content/uploads/2020/12/image.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [ ]:
TARGET_WAVS = [
    os.path.join(_DATA_DIR, "voxceleb1_test", "wav", "id10276", "5WOZXLMujwY", "00003.wav"),
    os.path.join(_DATA_DIR, "voxceleb1_test", "wav", "id10288", "dRr05CxK0jQ", "00001.wav"),
]

missing_wavs = [path for path in TARGET_WAVS if not os.path.isfile(path)]
if missing_wavs:
    raise FileNotFoundError("Не найдены аудиофрагменты вариантов: " + ", ".join(missing_wavs))

print("Подготовлены аудиофрагменты вариантов:")
for path in TARGET_WAVS:
    print(path)

In [ ]:
# Полный архив VoxCeleb1 для этой работы не распаковываем:
# в папке lab2/data оставлены только два фрагмента, соответствующие вариантам.
for path in TARGET_WAVS:
    print(os.path.relpath(path, _LAB_DIR))

**2. Выполнение процедуры высокочастотной фильтрации (преэмфазис) речевого сигнала**

В рамках настоящего пункта предлагается изучить процедуру предобработки речевых сигналов, получившую название преэмфазис. *Преэмфазис (pre-emphasis)* – применение фильтра верхних частот к сигналу до процедуры извлечения признаков для того, чтобы компенсировать тот факт, что обычно вокализованная речь содержит на низких частотах намного больше энергии, чем невокализованная речь на высоких. Обозначим через $x(n)$ обрабатываемый сигнал, а через $y(n)$ результат обработки, тогда процедура преэмфазиса может быть описана с помощью следующего выражения:

$$y(n) = x(n) + \alpha x(n-1).$$

Выражение выше представляет собой *линейное разностное уравнение с постоянными коэффициентами* и позволяет описать процедуру работы *нерекурсивного фильтра первого порядка с конечной импульсной характеристикой*. Параметр $\alpha$ является единственным параметром рассматриваемого фильтра. В случае, когда параметр $\alpha$ является отрицательным, рассматриваемый фильтр обладает свойствами *фильтра верхних частот*. Для выполнения процедуры преэмфазиса обычно параметр $\alpha = -0.97$.

В рамках настоящего пункта требуется:

1. Загрузить анализируемый речевой сигнал.

2. Выполнить процедуру преэмфазиса по отношению к загруженному речевому сигналу.

3. Сравнить спектры речевых сигналов до и после процедуры преэмфазиса.

Ниже — общий мел-банк и по очереди полный пайплайн признаков для каждого варианта.

In [ ]:
SPEAKER_IDS = ("id10276", "id10288")

with open(_META_PATH, "r") as f:
    list_lines = f.readlines()
p = Pool(1)
speaker_ids, genders, paths = zip(*p.map(split_meta_line, list_lines[1:]))
p.close()
path_by_id = dict(zip(speaker_ids, paths))

fbank = compute_fbank_filters(nfilt=40, sample_rate=16000, NFFT=512)

plt.figure(figsize=(15, 5))
plot_a = plt.subplot()
plt.subplots_adjust(wspace=0, hspace=1)
nfilt = fbank.shape[0]
for k in range(nfilt):
    plot_a.plot(fbank[k, :])
plot_a.set_xlabel("Frequency bins")
plot_a.set_ylabel("Amplitude")
plot_a.title.set_text("Fbank (общий мел-банк)")
plot_a.grid()
plt.show()

for SPEAKER_ID in SPEAKER_IDS:
    path_to_wav = path_by_id[SPEAKER_ID]
    print("Загрузка:", SPEAKER_ID, path_to_wav)

    signal, sample_rate = torchaudio.load(path_to_wav)
    signal = signal.numpy().squeeze(axis=0)
    signal = signal / np.abs(signal).max()

    signal = signal[0 : int(3.5 * sample_rate)]

    plt.figure(figsize=(15, 5))
    plt.suptitle(f"Диктор {SPEAKER_ID}: осциллограмма и спектрограмма (3.5 с)", y=1.02)
    plot_a = plt.subplot(211)
    plt.subplots_adjust(wspace=0, hspace=0.5)
    plot_a.plot(signal)
    plot_a.set_xlabel("n")
    plot_a.set_ylabel("x(n)")
    plot_a.grid()
    plot_b = plt.subplot(212)
    plot_b.specgram(signal, NFFT=1024, Fs=sample_rate, noverlap=900)
    plot_b.set_xlabel("Time, s")
    plot_b.set_ylabel("Frequency, Hz")
    plt.show()

    emphasized_signal = preemphasis(signal)

    plt.figure(figsize=(15, 5))
    plt.suptitle(f"Диктор {SPEAKER_ID}: спектрограммы до/после преэмфазиса", y=1.02)
    plot_a = plt.subplot(211)
    plt.subplots_adjust(wspace=0, hspace=1)
    plot_a.specgram(signal, NFFT=1024, Fs=sample_rate, noverlap=900)
    plot_a.set_xlabel("Time, s")
    plot_a.set_ylabel("Frequency, Hz")
    plot_a.title.set_text("Original signal")
    plot_b = plt.subplot(212)
    plot_b.specgram(emphasized_signal, NFFT=1024, Fs=sample_rate, noverlap=900)
    plot_b.set_xlabel("Time, s")
    plot_b.set_ylabel("Frequency, Hz")
    plot_b.title.set_text("Emphasized signal")
    plt.show()

    frames = framing(emphasized_signal)
    print(f"{SPEAKER_ID}: shape фреймов", frames.shape)

    pow_frames = power_spectrum(frames)

    filter_banks_features = compute_fbanks_features(pow_frames, fbank)

    plt.figure(figsize=(15, 5))
    plt.suptitle(f"Диктор {SPEAKER_ID}: сигнал и лог-энергии мел-банка", y=1.02)
    plt.subplots_adjust(wspace=0, hspace=1)
    plot_a = plt.subplot(211)
    plot_a.plot(signal)
    plot_a.set_xlabel("n")
    plot_a.set_ylabel("x(n)")
    plot_a.grid()
    plot_b = plt.subplot(212)
    plot_b.imshow(filter_banks_features.T, origin="lower")
    plot_b.set_xlabel("Time bins")
    plot_b.set_ylabel("Frequency band bins")
    plot_b.title.set_text("Fbank energies")
    plt.show()

    mfcc = compute_mfcc(filter_banks_features, num_ceps=20)

    fig = plt.figure(figsize=(15, 5))
    plt.suptitle(f"Диктор {SPEAKER_ID}: сигнал и MFCC", y=1.02)
    plt.subplots_adjust(wspace=0, hspace=1)
    plot_a = plt.subplot(211)
    plot_a.plot(signal)
    plot_a.set_xlabel("n")
    plot_a.set_ylabel("x(n)")
    plot_a.grid()
    plot_b = plt.subplot(212)
    im = plot_b.imshow(mfcc.T, origin="lower")
    divider = make_axes_locatable(plot_b)
    cax = divider.append_axes("right", size="1%", pad=0.05)
    plt.colorbar(im, cax=cax)
    plot_b.set_xlabel("Time bins")
    plot_b.set_ylabel("Coeffitients bins")
    plot_b.title.set_text("MFCCs")
    plt.show()

    mfcc_cmvn = mvn_floating(mfcc, 150, 150)
    filter_banks_features_mvn = mvn_floating(filter_banks_features, 150, 150)

    fig = plt.figure(figsize=(15, 5))
    plt.suptitle(f"Диктор {SPEAKER_ID}: нормализованные FBank и MFCC (CMVN)", y=1.02)
    plt.subplots_adjust(wspace=0, hspace=1)
    plot_b = plt.subplot(211)
    im_b = plot_b.imshow(filter_banks_features_mvn.T, origin="lower")
    divider = make_axes_locatable(plot_b)
    cax = divider.append_axes("right", size="1%", pad=0.05)
    plt.colorbar(im_b, cax=cax)
    plot_b.set_xlabel("Time bins")
    plot_b.set_ylabel("Coeffitients bins")
    plot_b.title.set_text("Normaized FBanks")
    plot_c = plt.subplot(212)
    im_c = plot_c.imshow(mfcc_cmvn.T, origin="lower")
    divider = make_axes_locatable(plot_c)
    cax = divider.append_axes("right", size="1%", pad=0.05)
    plt.colorbar(im_c, cax=cax)
    plot_c.set_xlabel("Time bins")
    plot_c.set_ylabel("Coeffitients bins")
    plot_c.title.set_text("Normaized MFCCs")
    plt.show()
    

**3. Построение распределений первых трёх компонент полученных акустических признаков для мужских и женских голосов**

Для проверки расчёта акустических признаков построим гистограммы распределения первых трёх компонент логарифмов энергий на выходе банка фильтров и мел-частотных кепстральных коэффициентов. В этой версии лабораторной в **./metadata/meta.txt** оставлены только два диктора, соответствующие вариантам: `id10276` (мужской голос) и `id10288` (женский голос).

Используя эти звукозаписи, выполним построение гистограмм первых 3 компонент MFB и MFCC отдельно для мужского и женского голосов.

In [ ]:
def compute_feats(signal):
    emphasized_signal = preemphasis(signal)
    frames = framing(emphasized_signal)
    pow_frames = power_spectrum(frames)
    filter_banks_features = compute_fbanks_features(pow_frames, fbank)
    mfcc = compute_mfcc(filter_banks_features, num_ceps=20)
    return mvn_floating(filter_banks_features, 150, 150), mvn_floating(mfcc, 150, 150)

male_fb_features = []
female_fb_features = []
male_mfcc_features = []
female_mfcc_features = []

for path_to_wav, gender in zip(paths, genders):
    signal, sample_rate = torchaudio.load(path_to_wav)
    signal = signal.numpy().squeeze(axis=0)
    signal = signal / np.abs(signal).max()

    filter_banks_mvn, mfcc_cmvn = compute_feats(signal)
    if gender == "m":
        male_fb_features.append(filter_banks_mvn)
        male_mfcc_features.append(mfcc_cmvn)
    else:
        female_fb_features.append(filter_banks_mvn)
        female_mfcc_features.append(mfcc_cmvn)

male_fb_features = np.concatenate(male_fb_features)
female_fb_features = np.concatenate(female_fb_features)
male_mfcc_features = np.concatenate(male_mfcc_features)
female_mfcc_features = np.concatenate(female_mfcc_features)

print("MFB male/female:", male_fb_features.shape, female_fb_features.shape)
print("MFCC male/female:", male_mfcc_features.shape, female_mfcc_features.shape)

for feature_name, male_features, female_features in [
    ("MFBs", male_fb_features, female_fb_features),
    ("MFCCs", male_mfcc_features, female_mfcc_features),
]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"Normalized histograms: first three {feature_name} components", y=1.05)
    for comp_number, ax in enumerate(axes):
        coeff_male = male_features[:, comp_number]
        coeff_female = female_features[:, comp_number]
        min_coeff = min(coeff_male.min(), coeff_female.min())
        max_coeff = max(coeff_male.max(), coeff_female.max())
        bins = int(sqrt(max(len(coeff_male), len(coeff_female))))

        ax.hist(coeff_male, bins, histtype="step", color="green", range=(min_coeff, max_coeff), density=True, label="male")
        ax.hist(coeff_female, bins, histtype="step", color="red", range=(min_coeff, max_coeff), density=True, label="female")
        ax.set_xlabel(f"{feature_name}, component {comp_number + 1}")
        ax.set_ylabel("Histogram value")
        ax.grid()
        ax.legend()
    plt.show()


**4. Контрольные вопросы**

1. Какие способы представления сигналов существуют?

2. Что такое спектр Фурье (амплитудный и фазовый)?

3. Что такое оконное преобразование Фурье?

4. Что такое спектрограмма?

5. Как выполнить процедуру преэмфазиса?

6. Описать процедуру вычисления акустических признаков.

7. Для каких целей выполняются процедуры нормализации и масштабирования акустических признаков?